In [15]:
from pathlib import Path
import tiktoken

import torch
from torch.utils.data import Dataset, DataLoader

In [16]:
path_to_data = Path().cwd().parents[1] / 'data' / 'train_data.txt'

In [17]:
with open(path_to_data, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f"Total number of characters: {len(raw_text)}")

Total number of characters: 438476740


In [18]:
# we define the dataset
# it will use the sliding window approach to create inputs and outputs
# it will have 4 parameters
# text, max_window_length, tokenizer, stride

# stride is how how much tokens the window will move in the next iteration

class GPTV0Dataset(Dataset):
    def __init__(self, text, max_window_length, tokenizer, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_window_length, stride):
            input_window = token_ids[i: i + max_window_length]
            target_window = token_ids[i + 1: i + max_window_length + 1]

            self.input_ids.append(torch.tensor(input_window))
            self.target_ids.append(torch.tensor(target_window))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [19]:
# now we define a function that creates the dataloader for the dataset
def create_dataloaderV0(text, max_window_length, tokenizer, stride, batch_size, shuffle, drop_last, num_workers):
    dataset = GPTV0Dataset(text, max_window_length, tokenizer, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [20]:
tokenizer = tiktoken.get_encoding('gpt2')

dataloader = create_dataloaderV0(raw_text[:10_000], max_window_length=256,      # first 10,000 characters of text
    tokenizer=tokenizer, stride=256, batch_size=1, shuffle=True, drop_last=True, num_workers=0)

In [21]:
data_iter = iter(dataloader)

first_batch = next(data_iter)
print(f"First batch input shape: {first_batch[0].shape}")
print(f"First batch input: {first_batch[0][0,:10]}")   # first 10 tokens of the first input sequence
print(f"First batch input decoded: {tokenizer.decode(first_batch[0][0,:10].tolist())}")

First batch input shape: torch.Size([1, 256])
First batch input: tensor([  286,   262, 40120,  1015, 18692,    13,   632,   373,   262,  8069])
First batch input decoded:  of the Mojave Desert. It was the 1970


In [22]:
# this is how many tokens the tokenizer has in its vocabulary already gpt2 bpe
print(f"Tokenizer vocabulary size: {tokenizer.n_vocab}")

Tokenizer vocabulary size: 50257
